In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

# ====================== 1. 读取数据 ======================
df = pd.read_excel("D:\\文成数据库\\Nb-Si数据库3.4-成分-性能.xlsx")

features_name = [
     "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "V", "W", "Ta", "Sn",
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "mean_Molar Magnetic Susceptibility",
    "mean_Neutron Cross Section",
    "mean_Neutron Mass Absorption",
    "mean_Space Group Number",
    "mean_Speed of Sound",
    "mean_A1 atomic number",

    "mean_A6 atomic weight",
    "mean_C1 temperature melting",
    "mean_C12 modulus Young",
    "mean_C2 temperature boiling",
    "mean_C3 enthalpy vaporization",
    "mean_C4 enthalpy melting",
    "mean_C5 enthalpy atomization",
    "mean_E13 Electron affinity",
    "mean_E2 electronegativity (Pauling)",

    "mean_Electrophilicity Index",

    "mean_Glawe Number",
    "mean_Mass Magnetic Susceptibility",

    "mean_S10 Lattice Constants a",
    "mean_S11 Lattice Constants b",
    "mean_S12 Lattice Constants c",
    "mean_S13 radii atomic (coordination number 12) (pm)",
    "mean_S3 radii covalent",
    "mean_Volume Magnetic Susceptibility",
    "mean_共价半径 Pyykko(Double Bond) (pm)",
    "mean_共价半径 Pyykko(Single Bond) (pm)",
    "mean_共价半径 Pyykko(Triple Bond) (pm)",
    "mean_密度",
    "mean_摩尔体积(cm3/mol)",
    "mean_摩尔热容(J/mol/k)",
    "mean_比热容J/g/k",
    "mean_热导率W/(mk)",
    "mean_热膨胀(1/k)",
    "mean_电导率(MS/m)",
    "mean_电阻率(mΩ)",
    "mean_莫氏硬度(MPa)",
    "mean_金属半径 12 Neighbors(pm)",
    "mean_金属半径(pm)",
    "var_Molar Magnetic Susceptibility",
    "var_Neutron Cross Section",
    "var_Neutron Mass Absorption",
    "var_Space Group Number",
    "var_Speed of Sound",
    "var_A1 atomic number",

    "var_A6 atomic weight",
    "var_C1 temperature melting",
    "var_C12 modulus Young",
    "var_C2 temperature boiling",
    "var_C3 enthalpy vaporization",
    "var_C4 enthalpy melting",
    "var_C5 enthalpy atomization",
    "var_E13 Electron affinity",
    "var_E2 electronegativity (Pauling)",

    "var_Electrophilicity Index",

    "var_Glawe Number",
    "var_Mass Magnetic Susceptibility",

    "var_S10 Lattice Constants a",
    "var_S11 Lattice Constants b",
    "var_S12 Lattice Constants c",
    "var_S13 radii atomic (coordination number 12) (pm)",
    "var_S3 radii covalent",
    "var_Volume Magnetic Susceptibility",
    "var_共价半径 Pyykko(Double Bond) (pm)",
    "var_共价半径 Pyykko(Single Bond) (pm)",
    "var_共价半径 Pyykko(Triple Bond) (pm)",
    "var_密度",
    "var_摩尔体积(cm3/mol)",
    "var_摩尔热容(J/mol/k)",
    "var_比热容J/g/k",
    "var_热导率W/(mk)",
    "var_热膨胀(1/k)",
    "var_电导率(MS/m)",
    "var_电阻率(mΩ)",
    "var_莫氏硬度(MPa)",
    "var_金属半径 12 Neighbors(pm)",
    "var_金属半径(pm)"
]

target_col = 'KQ'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# -------------- 提取X与y --------------
X_all = df[features_name].values
y_all = df[target_col].values
y_all_np = np.array(y_all, dtype=float)

# ==== 创建可视化保存路径 ====
visualization_dir = "D:\\文成数据库\\3.16R2 rmse mae可视化"
if not os.path.exists(visualization_dir):
    os.makedirs(visualization_dir)

# ====================== 2. 单次标准化 (修改后) ======================
# 只对原始数据标准化一次，用于特征选择
scaler_for_selection = StandardScaler()
X_all_scaled = scaler_for_selection.fit_transform(X_all)

n_samples, n_features = X_all_scaled.shape

# ====================== 3. PCC 去除冗余(特征-特征) ======================
pcc_matrix = np.zeros((n_features, n_features))
for i in range(n_features):
    for j in range(i + 1, n_features):
        pcc_val, _ = pearsonr(X_all_scaled[:, i], X_all_scaled[:, j])
        pcc_matrix[i, j] = pcc_val
        pcc_matrix[j, i] = pcc_val

pcc_threshold = 0.95
redundant_features = set()
redundancy_logs = []

print("### PCC 去除冗余特征过程：")
print("| 冗余特征对 | 特征对 PCC 值 | 移除特征 | 移除特征与目标变量 PCC 值 |")
print("| --- | --- | --- | --- |")

for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(pcc_matrix[i, j]) > pcc_threshold:
            # 比较两个特征与目标变量的相关性
            pcc_i, _ = pearsonr(X_all_scaled[:, i], y_all_np)
            pcc_j, _ = pearsonr(X_all_scaled[:, j], y_all_np)
            if abs(pcc_i) < abs(pcc_j):
                redundant_features.add(i)
                removed_feature = features_name[i]
                removed_pcc = pcc_i
            else:
                redundant_features.add(j)
                removed_feature = features_name[j]
                removed_pcc = pcc_j
            log = {
                "冗余特征对": f"{features_name[i]} 和 {features_name[j]}",
                "特征对 PCC 值": pcc_matrix[i, j],
                "移除特征": removed_feature,
                "移除特征与目标变量 PCC 值": removed_pcc
            }
            redundancy_logs.append(log)
            print(f"| {features_name[i]} 和 {features_name[j]} | {pcc_matrix[i, j]:.4f} | {removed_feature} | {removed_pcc:.4f} |")

non_redundant_indices = [i for i in range(n_features) if i not in redundant_features]
X_nonredundant = X_all_scaled[:, non_redundant_indices]
nr_features_name = [features_name[i] for i in non_redundant_indices]

# ====================== 4. 过滤法：PCC + MIC (与目标) ======================
#  4.1 计算与目标的Pearson相关(绝对值大于一定阈值)
corr_threshold = 0.2
pcc_with_target = []
for i in range(X_nonredundant.shape[1]):
    p_val, _ = pearsonr(X_nonredundant[:, i], y_all_np)
    pcc_with_target.append(abs(p_val))

pcc_indices = [i for i, v in enumerate(pcc_with_target) if v > corr_threshold]

#  4.2 计算与目标的互信息(MIC)
mic_values = mutual_info_regression(X_nonredundant, y_all_np)
mic_threshold = 0.2
mic_indices = [i for i, v in enumerate(mic_values) if v > mic_threshold]

#  4.3 合并策略(交集)
selected_indices_filter = set(pcc_indices).intersection(set(mic_indices))
selected_indices_filter = sorted(list(selected_indices_filter))

filtered_features = [nr_features_name[i] for i in selected_indices_filter]
X_filter = X_nonredundant[:, selected_indices_filter]

print("\n=== 过滤法(PCC+MIC)之后保留的特征 ===")
print(filtered_features)

# ====================== 5. 包装法(RFE + ElasticNet) ======================
#  以ElasticNet作为基模型进行RFE
n_features_to_select = 6
elasticnet_estimator = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=0)
selector = RFE(estimator=elasticnet_estimator, n_features_to_select=n_features_to_select)
selector = selector.fit(X_filter, y_all_np)

selected_mask_rfe = selector.support_
final_features_rfe = [filtered_features[i] for i, sel in enumerate(selected_mask_rfe) if sel]

print("\n=== RFE(ElasticNet)后保留的最终6个特征 ===")
print(final_features_rfe)

# ====================== 6. 提取最终特征并标准化 (修改后) ======================
# 从原始数据中提取最终选择的特征
X_final_original = df[final_features_rfe].values

# 对最终特征进行标准化(这是唯一需要保存的标准化器)
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final_original)

# 为神经网络构建特征和目标数据
feature_rfe = pd.DataFrame(X_final_scaled, columns=final_features_rfe)
target = pd.Series(y_all_np, name=target_col)

# ====================== 7. 定义模型 ======================
class MyModel(nn.Module):
    def __init__(self, input_dim=12):
        super(MyModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 450)
        self.bn1 = nn.BatchNorm1d(450)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(450, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

my_nn = MyModel(len(final_features_rfe)).to(device)

# ====================== 8. 定义损失函数与优化器 ======================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(my_nn.parameters(), lr=0.01, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

# ====================== 9. 日志记录相关变量 ======================
train_loss_history = []
test_loss_history = []
train_r2_history = []
test_r2_history = []
train_rmse_history = []
test_rmse_history = []
train_mae_history = []
test_mae_history = []

rolling_train_r2 = []
rolling_test_r2 = []
rolling_train_rmse = []
rolling_test_rmse = []
rolling_train_mae = []
rolling_test_mae = []

# 存储最终模型评估的真实值和预测值
final_train_true = []
final_train_pred = []
final_test_true = []
final_test_pred = []

# 新增：存储最终训练集和测试集的索引
final_train_indices = []
final_test_indices = []

avg_train_r2 = None
avg_test_r2 = None
avg_train_rmse = None
avg_test_rmse = None
avg_train_mae = None
avg_test_mae = None
output_step = 0

# 设置一个标志变量，用于跟踪是否成功完成了训练并保存了预测值
prediction_data_saved = False

# ====================== 10. 训练循环 ======================
for epoch in range(1000000):
    # 每轮随机选一个种子来拆分数据
    i = random.randint(0, 4294967295)

    # 修改：使用train_test_split时返回索引而不是拆分后的数据
    train_indices, test_indices, _, _ = train_test_split(
        np.arange(len(feature_rfe)), np.arange(len(feature_rfe)), 
        test_size=0.2, random_state=i
    )
    
    # 通过索引获取训练集和测试集数据
    x_train_np = feature_rfe.values[train_indices]
    y_train_np = target.values[train_indices]
    x_test_np = feature_rfe.values[test_indices]
    y_test_np = target.values[test_indices]

    train_x = torch.from_numpy(x_train_np.astype(np.float32)).to(device)
    train_y = torch.from_numpy(y_train_np.astype(np.float32)).to(device).view(-1, 1)
    test_x = torch.from_numpy(x_test_np.astype(np.float32)).to(device)
    test_y = torch.from_numpy(y_test_np.astype(np.float32)).to(device).view(-1, 1)

    # ---------- 前向传播、反向传播 ----------
    my_nn.train()
    optimizer.zero_grad()
    outputs = my_nn(train_x)
    loss = criterion(outputs, train_y)
    loss.backward()
    optimizer.step()
    train_loss = loss.item()

    # 更新学习率
    scheduler.step()

    # ---------- 评估 ----------
    my_nn.eval()
    with torch.no_grad():
        outputs_train = my_nn(train_x)
        r2_train = r2_score(y_train_np, outputs_train.cpu().numpy())
        rmse_train = np.sqrt(mean_squared_error(y_train_np, outputs_train.cpu().numpy()))
        mae_train = mean_absolute_error(y_train_np, outputs_train.cpu().numpy())

        outputs_test = my_nn(test_x)
        r2_test = r2_score(y_test_np, outputs_test.cpu().numpy())
        rmse_test = np.sqrt(mean_squared_error(y_test_np, outputs_test.cpu().numpy()))
        mae_test = mean_absolute_error(y_test_np, outputs_test.cpu().numpy())
        test_loss = criterion(outputs_test, test_y).item()

    # 记录Loss，用于可视化
    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)

    # ---------- 滚动平均 ----------
    rolling_train_r2.append(r2_train)
    rolling_test_r2.append(r2_test)
    rolling_train_rmse.append(rmse_train)
    rolling_test_rmse.append(rmse_test)
    rolling_train_mae.append(mae_train)
    rolling_test_mae.append(mae_test)

    # 每 5 次迭代计算一次平均
    if len(rolling_train_r2) == 5:
        avg_train_r2 = np.mean(rolling_train_r2)
        avg_test_r2 = np.mean(rolling_test_r2)
        avg_train_rmse = np.mean(rolling_train_rmse)
        avg_test_rmse = np.mean(rolling_test_rmse)
        avg_train_mae = np.mean(rolling_train_mae)
        avg_test_mae = np.mean(rolling_test_mae)

        train_r2_history.append(avg_train_r2)
        test_r2_history.append(avg_test_r2)
        train_rmse_history.append(avg_train_rmse)
        test_rmse_history.append(avg_test_rmse)
        train_mae_history.append(avg_train_mae)
        test_mae_history.append(avg_test_mae)

        # 清空滚动
        rolling_train_r2 = []
        rolling_test_r2 = []
        rolling_train_rmse = []
        rolling_test_rmse = []
        rolling_train_mae = []
        rolling_test_mae = []

        output_step += 1

        # 每累计10次（即50次训练）打印一次
        if output_step == 10:
            print(
                f"次数: {epoch} | 训练集 R2: {avg_train_r2:.4f} | 训练集 RMSE: {avg_train_rmse:.4f} | 训练集 MAE: {avg_train_mae:.4f} "
                f"| 测试集 R2: {avg_test_r2:.4f} | 测试集 RMSE: {avg_test_rmse:.4f} | 测试集 MAE: {avg_test_mae:.4f} "
                f"| LR: {optimizer.param_groups[0]['lr']}"
            )

            # 保存当前的训练和测试数据的真实值和预测值（用于后续绘图）
            final_train_true = y_train_np.flatten()
            final_train_pred = outputs_train.cpu().numpy().flatten()
            final_test_true = y_test_np.flatten()
            final_test_pred = outputs_test.cpu().numpy().flatten()
            
            # 保存当前的训练集和测试集索引
            final_train_indices = train_indices
            final_test_indices = test_indices
            
            # 更新预测数据已保存标志
            prediction_data_saved = True

            # 当达到保存条件时，结束训练
            if ((avg_train_r2 > 0.95 and avg_test_r2 > 0.91)
                and (avg_train_rmse < 0.9 and avg_test_rmse < 0.9)
            ):
                print(f"模型在第 {epoch} 轮训练后达到条件，训练结束！| 随机种子： {i}")
                break  # 终止训练

            output_step = 0

else:
    print("训练结束，未达到模型保存条件。")
    if train_r2_history and test_r2_history and train_rmse_history and test_rmse_history:
        final_avg_train_r2 = train_r2_history[-1]
        final_avg_test_r2 = test_r2_history[-1]
        final_avg_train_rmse = train_rmse_history[-1]
        final_avg_test_rmse = test_rmse_history[-1]
        final_avg_train_mae = train_mae_history[-1]
        final_avg_test_mae = test_mae_history[-1]
        print(f"最终训练集 R2: {final_avg_train_r2:.4f} | 最终训练集 RMSE: {final_avg_train_rmse:.4f} | 最终训练集 MAE: {final_avg_train_mae:.4f}")
        print(f"最终测试集 R2: {final_avg_test_r2:.4f} | 最终测试集 RMSE: {final_avg_test_rmse:.4f} | 最终测试集 MAE: {final_avg_test_mae:.4f}")

# ====================== 11. 可视化与保存 ======================

# 1. 创建专门的散点图可视化（类似于图片中的样式）
def create_scatter_plot(true_values_train, pred_values_train, 
                        true_values_test, pred_values_test, 
                        model_name, save_path):
    """
    创建散点图：真实值与预测值的对比
    """
    # 计算训练集和测试集的指标
    r2_train = r2_score(true_values_train, pred_values_train)
    rmse_train = np.sqrt(mean_squared_error(true_values_train, pred_values_train))
    mae_train = mean_absolute_error(true_values_train, pred_values_train)
    
    r2_test = r2_score(true_values_test, pred_values_test)
    rmse_test = np.sqrt(mean_squared_error(true_values_test, pred_values_test))
    mae_test = mean_absolute_error(true_values_test, pred_values_test)
    
    # 找出所有数据点的最小值和最大值，用于设置轴的范围
    all_values = np.concatenate([
        true_values_train, pred_values_train,
        true_values_test, pred_values_test
    ])
    min_val = np.min(all_values)
    max_val = np.max(all_values)
    
    # 创建子图
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 绘制对角线（理想预测线）
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.75)
    
    # 绘制训练集散点图（蓝色点）
    ax.scatter(true_values_train, pred_values_train, color='blue', alpha=0.6, 
               edgecolor='black', linewidth=0.5, label='Train')
    
    # 绘制测试集散点图（红色点）
    ax.scatter(true_values_test, pred_values_test, color='#FF69B4', alpha=0.6,
               edgecolor='black', linewidth=0.5, label='Test')
    
    # 添加图例、标题和标签
    ax.set_xlabel('True value', fontsize=12)
    ax.set_ylabel('Predicted value', fontsize=12)
    ax.set_title(model_name, fontsize=14)
    
    # 保持正方形比例
    ax.set_aspect('equal')
    
    # 设置轴的范围略大于数据点的范围
    margin = (max_val - min_val) * 0.05
    ax.set_xlim(min_val - margin, max_val + margin)
    ax.set_ylim(min_val - margin, max_val + margin)
    
    # 添加网格
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # 添加指标文本（左上角训练集，右下角测试集）
    # 训练集指标（蓝色文本）
    ax.text(0.05, 0.95, 
           f'Train\nMAE={mae_train:.3f}\nRMSE={rmse_train:.3f}\nR²={r2_train:.3f}',
           transform=ax.transAxes, fontsize=10, color='blue',
           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 测试集指标（粉色文本）
    ax.text(0.75, 0.05, 
           f'Test\nMAE={mae_test:.3f}\nRMSE={rmse_test:.3f}\nR²={r2_test:.3f}',
           transform=ax.transAxes, fontsize=10, color='#FF1493',
           verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 保存图片
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# 检查是否需要获取预测数据
if not prediction_data_saved or len(final_train_true) == 0 or len(final_test_true) == 0 or len(final_train_indices) == 0 or len(final_test_indices) == 0:
    print("未找到有效的预测数据，使用最后一次训练的模型进行预测...")
    
    # 重新划分训练集和测试集
    train_indices, test_indices, _, _ = train_test_split(
        np.arange(len(feature_rfe)), np.arange(len(feature_rfe)), 
        test_size=0.2, random_state=42  # 使用固定种子以便复现
    )
    
    # 获取训练集和测试集数据
    x_train_np = feature_rfe.values[train_indices]
    y_train_np = target.values[train_indices]
    x_test_np = feature_rfe.values[test_indices]
    y_test_np = target.values[test_indices]
    
    train_x = torch.from_numpy(x_train_np.astype(np.float32)).to(device)
    test_x = torch.from_numpy(x_test_np.astype(np.float32)).to(device)
    
    # 确保我们有有效的数据进行操作
    my_nn.eval()
    with torch.no_grad():
        outputs_train = my_nn(train_x)
        outputs_test = my_nn(test_x)
        
        final_train_true = y_train_np.flatten()
        final_train_pred = outputs_train.cpu().numpy().flatten()
        final_test_true = y_test_np.flatten()
        final_test_pred = outputs_test.cpu().numpy().flatten()
        
        # 保存索引
        final_train_indices = train_indices
        final_test_indices = test_indices

# 2. 创建散点图
create_scatter_plot(
    final_train_true, final_train_pred,
    final_test_true, final_test_pred,
    'Neural Network', 
    os.path.join(visualization_dir, 'NN_散点图.png')
)

# 3. 创建多个子图的组合图（一行三列）
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
model_names = ['NN (G1)', 'NN (H1)', 'NN (I1)']

for i, ax in enumerate(axs):
    # 绘制对角线
    all_values = np.concatenate([final_train_true, final_train_pred, 
                                final_test_true, final_test_pred])
    min_val = np.min(all_values)
    max_val = np.max(all_values)
    margin = (max_val - min_val) * 0.05
    
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.75)
    
    # 绘制训练集散点图
    ax.scatter(final_train_true, final_train_pred, color='blue', alpha=0.6, 
              edgecolor='black', linewidth=0.5, s=30)
    
    # 绘制测试集散点图
    ax.scatter(final_test_true, final_test_pred, color='#FF69B4', alpha=0.6,
              edgecolor='black', linewidth=0.5, s=30)
    
    # 设置标题和标签
    ax.set_title(model_names[i], fontsize=12)
    ax.set_xlabel('True value', fontsize=10)
    ax.set_ylabel('Predicted value', fontsize=10)
    
    # 保持正方形比例
    ax.set_aspect('equal')
    ax.set_xlim(min_val - margin, max_val + margin)
    ax.set_ylim(min_val - margin, max_val + margin)
    
    # 添加网格
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # 添加指标文本
    r2_train = r2_score(final_train_true, final_train_pred)
    rmse_train = np.sqrt(mean_squared_error(final_train_true, final_train_pred))
    mae_train = mean_absolute_error(final_train_true, final_train_pred)
    
    r2_test = r2_score(final_test_true, final_test_pred)
    rmse_test = np.sqrt(mean_squared_error(final_test_true, final_test_pred))
    mae_test = mean_absolute_error(final_test_true, final_test_pred)
    
    # 训练集指标
    ax.text(0.05, 0.95, 
           f'Train\nMAE={mae_train:.3f}\nRMSE={rmse_train:.3f}\nR²={r2_train:.3f}',
           transform=ax.transAxes, fontsize=8, color='blue',
           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # 测试集指标
    ax.text(0.75, 0.05, 
           f'Test\nMAE={mae_test:.3f}\nRMSE={rmse_test:.3f}\nR²={r2_test:.3f}',
           transform=ax.transAxes, fontsize=8, color='#FF1493',
           verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(visualization_dir, '多模型散点图比较.png'), dpi=300)
plt.close()

# 4. 保存训练指标数据
metrics_df = pd.DataFrame({
    'train_r2': train_r2_history,
    'test_r2': test_r2_history,
    'train_rmse': train_rmse_history,
    'test_rmse': test_rmse_history,
    'train_mae': train_mae_history,
    'test_mae': test_mae_history
})
metrics_df.to_excel(os.path.join(visualization_dir, '训练指标数据.xlsx'), index=False)

# 5. 修改: 将训练集和测试集预测结果放在一个表格中
# 获取两个数据集中的最小长度
min_length = min(len(final_train_true), len(final_test_true))

# 创建包含训练集和测试集数据的DataFrame
combined_df = pd.DataFrame({
    '训练集真实值': final_train_true[:min_length],
    '训练集预测值': final_train_pred[:min_length],
    '测试集真实值': final_test_true[:min_length],
    '测试集预测值': final_test_pred[:min_length]
})

# 保存合并后的结果
combined_df.to_excel(os.path.join(visualization_dir, '预测结果对比.xlsx'), index=False)
print("训练集和测试集预测结果已合并保存到一个表格中")

# 6. 新增: 对整个数据库计算MAPE, MRE, MRE+SD并可视化
def calculate_error_metrics(true_values, pred_values):
    """计算各种误差指标"""
    # 避免除以0的问题
    epsilon = 1e-10
    
    # MAPE (平均绝对百分比误差)
    mape = np.mean(np.abs((true_values - pred_values) / (np.abs(true_values) + epsilon))) * 100
    
    # MRE (平均相对误差)
    re = (pred_values - true_values) / (np.abs(true_values) + epsilon) * 100
    mre = np.mean(re)
    
    # MRE的标准差
    re_std = np.std(re)
    
    # MRE+SD
    mre_plus_sd = mre + re_std
    
    return mape, mre, re_std, mre_plus_sd

# 计算所有数据的预测值
my_nn.eval()
all_features_tensor = torch.from_numpy(feature_rfe.values.astype(np.float32)).to(device)
all_targets = target.values

with torch.no_grad():
    all_predictions = my_nn(all_features_tensor).cpu().numpy().flatten()

# 计算各种误差指标
all_mape, all_mre, all_re_std, all_mre_plus_sd = calculate_error_metrics(all_targets, all_predictions)

# 训练集和测试集的指标
train_mape, train_mre, train_re_std, train_mre_plus_sd = calculate_error_metrics(final_train_true, final_train_pred)
test_mape, test_mre, test_re_std, test_mre_plus_sd = calculate_error_metrics(final_test_true, final_test_pred)

# 创建误差指标可视化
fig, ax = plt.subplots(figsize=(10, 6))

# 数据
categories = ['全部数据', '训练集', '测试集']
mape_values = [all_mape, train_mape, test_mape]
mre_values = [all_mre, train_mre, test_mre]
mre_plus_sd_values = [all_mre_plus_sd, train_mre_plus_sd, test_mre_plus_sd]

# 设置X轴位置
x = np.arange(len(categories))
width = 0.25

# 绘制柱状图
bars1 = ax.bar(x - width, mape_values, width, label='MAPE (%)', color='skyblue', edgecolor='black')
bars2 = ax.bar(x, mre_values, width, label='MRE (%)', color='lightgreen', edgecolor='black')
bars3 = ax.bar(x + width, mre_plus_sd_values, width, label='MRE+SD (%)', color='salmon', edgecolor='black')

# 添加文本标签、图例和标题
ax.set_xlabel('数据集')
ax.set_ylabel('误差 (%)')
ax.set_title('预测误差指标对比')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# 在柱状图上添加数值标签
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

plt.tight_layout()
plt.savefig(os.path.join(visualization_dir, '误差指标对比.png'), dpi=300)
plt.close()

# 7. 修改：创建带有数据集分类的所有数据点误差分析
# 创建一个表示每个样本属于哪个数据集的数组
dataset_labels = np.ones(len(all_targets), dtype=int) * -1  # 初始化为-1，表示未知
dataset_labels[final_train_indices] = 0  # 训练集标记为0
dataset_labels[final_test_indices] = 1  # 测试集标记为1

# 转换数字标签为字符串标签
dataset_names = np.array(['未知'] * len(all_targets), dtype=object)
dataset_names[dataset_labels == 0] = '训练集'
dataset_names[dataset_labels == 1] = '测试集'

# 计算每个数据点的误差
absolute_errors = np.abs(all_targets - all_predictions)
relative_errors = np.abs((all_targets - all_predictions) / (np.abs(all_targets) + 1e-10)) * 100
mre_errors = ((all_predictions - all_targets) / (np.abs(all_targets) + 1e-10)) * 100

# 创建DataFrame
all_results_df = pd.DataFrame({
    '数据集': dataset_names,
    '真实值': all_targets,
    '预测值': all_predictions,
    '绝对误差': absolute_errors,
    '相对误差(%)': relative_errors,
    'MAPE(%)': relative_errors,
    'MRE(%)': mre_errors
})

# 保存完整结果
all_results_df.to_excel(os.path.join(visualization_dir, '所有数据点误差分析.xlsx'), index=False)

# 按数据集分组计算汇总统计量
grouped_stats = all_results_df.groupby('数据集').agg({
    'MAPE(%)': 'mean',
    'MRE(%)': ['mean', 'std']
}).reset_index()

# 计算每个组的MRE+SD
grouped_stats['MRE+SD'] = grouped_stats[('MRE(%)', 'mean')] + grouped_stats[('MRE(%)', 'std')]

# 重命名列以便于阅读
grouped_stats.columns = ['数据集', 'MAPE(%)', 'MRE(%)', 'MRE的标准差', 'MRE+SD(%)']

# 添加整体数据指标
overall_row = pd.DataFrame({
    '数据集': ['全部数据'],
    'MAPE(%)': [all_mape],
    'MRE(%)': [all_mre],
    'MRE的标准差': [all_re_std],
    'MRE+SD(%)': [all_mre_plus_sd]
})

# 合并汇总数据
summary_df = pd.concat([overall_row, grouped_stats])

# 保存汇总统计结果
summary_df.to_excel(os.path.join(visualization_dir, '误差指标汇总.xlsx'), index=False)

print("已完成MAPE, MRE, MRE+SD的计算和可视化，结果已按数据集分类保存到Excel文件中")

# 8. 创建不同数据集预测误差的分布对比图
plt.figure(figsize=(12, 6))

# 训练集和测试集的相对误差数据
train_errors = all_results_df[all_results_df['数据集'] == '训练集']['相对误差(%)']
test_errors = all_results_df[all_results_df['数据集'] == '测试集']['相对误差(%)']

# 设置柱状图的参数
bins = 20
alpha = 0.7

# 创建训练集和测试集的误差直方图
plt.hist(train_errors, bins=bins, alpha=alpha, color='blue', label='训练集')
plt.hist(test_errors, bins=bins, alpha=alpha, color='#FF69B4', label='测试集')

# 添加图表元素
plt.xlabel('相对误差 (%)')
plt.ylabel('频数')
plt.title('训练集与测试集预测误差分布对比')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# 保存图表
plt.tight_layout()
plt.savefig(os.path.join(visualization_dir, '误差分布对比.png'), dpi=300)
plt.close()

print("已创建训练集与测试集误差分布对比图")

# 打印出最终特征列表供第二个程序使用
print("\n=== 最终模型使用的6个特征 ===")
for i, feature in enumerate(final_features_rfe):
    print(f"{i+1}. {feature}")
print("\n请确保第二个程序(预测程序)计算这些特征并按此顺序排列")
# ====================== 12. 可视化训练集与测试集特征分布 ======================
import seaborn as sns

# 创建 feature_rfe 的副本用于可视化
feature_rfe_copy = feature_rfe.copy()
feature_rfe_copy['dataset'] = 'unknown'
feature_rfe_copy.loc[final_train_indices, 'dataset'] = 'train'
feature_rfe_copy.loc[final_test_indices, 'dataset'] = 'test'

# 替换列名（只修改指定两个）
rename_dict = {
    'mean_C4 enthalpy melting': 'M-F4',
    'mean_Electrophilicity Index': 'M-B3'
}
feature_rfe_copy.rename(columns=rename_dict, inplace=True)

# 设置可视化风格
sns.set(style="ticks")

# 绘制 Pair Plot 图（对角线上显示直方图）
pair_plot = sns.pairplot(
    feature_rfe_copy,
    hue='dataset',
    diag_kind='hist',  # 可改为 'kde'
    plot_kws={'alpha': 0.6}
)

# 设置标题
pair_plot.fig.suptitle("训练集与测试集特征分布对比", y=1.02)

# 保存图片到指定目录
pairplot_path = os.path.join(visualization_dir, '训练集_vs_测试集_特征分布对比.png')
pair_plot.savefig(pairplot_path, dpi=300)
plt.close()

print(f"已保存训练集和测试集的特征分布对比图：{pairplot_path}")

### PCC 去除冗余特征过程：
| 冗余特征对 | 特征对 PCC 值 | 移除特征 | 移除特征与目标变量 PCC 值 |
| --- | --- | --- | --- |
| Nb 和 ΔSmix | -0.9717 | Nb | 0.0426 |
| Si 和 ΔHmix | -0.9856 | Si | -0.6229 |
| Si 和 Δa | 0.9587 | Δa | -0.6008 |
| Si 和 mean_电阻率(mΩ) | 1.0000 | mean_电阻率(mΩ) | -0.6227 |
| Si 和 var_Glawe Number | 0.9716 | var_Glawe Number | -0.6145 |
| Si 和 var_S13 radii atomic (coordination number 12) (pm) | 0.9593 | var_S13 radii atomic (coordination number 12) (pm) | -0.5872 |
| Si 和 var_S3 radii covalent | 0.9644 | var_S3 radii covalent | -0.6003 |
| Si 和 var_摩尔热容(J/mol/k) | 0.9840 | var_摩尔热容(J/mol/k) | -0.6118 |
| Si 和 var_电阻率(mΩ) | 0.9981 | Si | -0.6229 |
| Hf 和 mean_Neutron Cross Section | 0.9786 | mean_Neutron Cross Section | -0.0449 |
| Hf 和 var_Neutron Cross Section | 0.9992 | Hf | -0.1141 |
| ΔHmix 和 Δa | -0.9731 | Δa | -0.6008 |
| ΔHmix 和 mean_电阻率(mΩ) | -0.9856 | mean_电阻率(mΩ) | -0.6227 |
| ΔHmix 和 var_Glawe Number | -0.9888 | var_Glawe Number | -0.6145 |
| ΔHmix 和 var_S13 radii atomic (coordination n